# Semantic Extractor Training

Enable GPU, attach the private semantic training dataset, and run all cells.
The notebook writes only safe artifacts to `/kaggle/working/`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile

RUN_ID = "__RUN_ID__"
EXPECTED_GIT_COMMIT = "__EXPECTED_GIT_COMMIT__"
RUN_ROOT = Path('/kaggle/working/smoke_runs') / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

notebook_started_path = RUN_ROOT / 'notebook_started.json'
notebook_started_path.write_text(
    json.dumps(
        {
            'run_id': RUN_ID,
            'expected_git_commit': EXPECTED_GIT_COMMIT,
            'executed_git_commit': None,
            'timestamp': time.time(),
            'pid': os.getpid(),
            'python_version': sys.version,
            'smoke_mode': True,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)
(RUN_ROOT / 'runner_metadata.json').write_text(
    json.dumps(
        {
            'run_id': RUN_ID,
            'expected_git_commit': EXPECTED_GIT_COMMIT,
            'timestamp': time.time(),
            'pid': os.getpid(),
        },
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)

repo_zip = Path('/kaggle/working/data_analysis_LLM.zip')
repo_root = Path('/kaggle/working/data_analysis_LLM')
urllib.request.urlretrieve('https://github.com/PritishMete/data_analysis_LLM/archive/refs/heads/main.zip', repo_zip)
with tempfile.TemporaryDirectory(dir='/kaggle/working') as temp_dir:
    with zipfile.ZipFile(repo_zip, 'r') as archive:
        archive.extractall(temp_dir)
    extracted = next(Path(temp_dir).glob('data_analysis_LLM-*'))
    if repo_root.exists():
        shutil.rmtree(repo_root)
    if extracted.is_dir():
        shutil.move(str(extracted), str(repo_root))
executed_git_commit = subprocess.run([
    'git', 'rev-parse', 'HEAD'
], cwd=repo_root, capture_output=True, text=True, check=True, encoding='utf-8', errors='replace').stdout.strip()
started_payload = json.loads(notebook_started_path.read_text(encoding='utf-8'))
started_payload['executed_git_commit'] = executed_git_commit
notebook_started_path.write_text(json.dumps(started_payload, indent=2, sort_keys=True), encoding='utf-8')
(RUN_ROOT / 'runner_metadata.json').write_text(
    json.dumps(
        {
            'run_id': RUN_ID,
            'expected_git_commit': EXPECTED_GIT_COMMIT,
            'executed_git_commit': executed_git_commit,
            'timestamp': time.time(),
            'pid': os.getpid(),
        },
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)
os.environ['KAGGLE_SMOKE_RUN_ID'] = RUN_ID
os.environ['KAGGLE_EXPECTED_GIT_COMMIT'] = EXPECTED_GIT_COMMIT
subprocess.run([
    sys.executable,
    str(repo_root / 'kaggle' / 'bootstrap_environment.py'),
    '--output-root', '/kaggle/working',
    '--run-id', RUN_ID,
], check=True)
bootstrap_report = json.loads((RUN_ROOT / 'dependency_install_result.json').read_text())
bootstrap_pid = bootstrap_report.get('bootstrap_pid') or 0
training = subprocess.run([
    sys.executable,
    str(repo_root / 'kaggle' / 'execute_smoke_training.py'),
    '--output-root', '/kaggle/working',
    '--bootstrap-pid', str(bootstrap_pid),
    '--run-id', RUN_ID,
    '--expected-git-commit', EXPECTED_GIT_COMMIT,
], check=True)
raise SystemExit(training.returncode)
